# Positional Encodings

In the previous chapters we represented each word by a vector (an embedding), e.g. learned with Word2Vec. The Transformer architecture introduced in "Attention Is All You Need" [@VASWANI2017AttentionAllYou] processes all tokens of a sentence *at the same time* with self-attention instead of reading them one after another. This chapter explains how the model still learns *where* each word is in the sentence: by adding a **positional encoding** to each word embedding.

## Why do we need positional information?

Consider the two sentences

- "Alice was tired"
- "tired was Alice"

An embedding lookup assigns the *same* vector to "Alice" no matter where the word appears. Self-attention computes each output as a weighted sum over *all* tokens, where the weights depend only on the vectors themselves. If we shuffle the tokens, we get the same vectors back, only in a different order: nothing in the input tells the model which word came first.

The cell below shows this with a toy embedding lookup.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

toy_vocabulary = ["Alice", "was", "tired"]
toy_embeddings = {word: np.round(np.random.randn(3), 2) for word in toy_vocabulary}

sentence_a = ["Alice", "was", "tired"]
sentence_b = ["tired", "was", "Alice"]

print("Embeddings for:", " ".join(sentence_a))
print(pd.DataFrame([toy_embeddings[w] for w in sentence_a], index=sentence_a))
print("\nEmbeddings for:", " ".join(sentence_b))
print(pd.DataFrame([toy_embeddings[w] for w in sentence_b], index=sentence_b))

Embeddings for: Alice was tired
          0     1     2
Alice  0.50 -0.14  0.65
was    1.52 -0.23 -0.23
tired  1.58  0.77 -0.47

Embeddings for: tired was Alice
          0     1     2
tired  1.58  0.77 -0.47
was    1.52 -0.23 -0.23
Alice  0.50 -0.14  0.65


Both tables contain exactly the same rows. The idea of positional encodings is to add a vector $PE_{pos}$ that depends *only on the position* $pos$ of a token to its word embedding $x$:

$$
\text{input to the Transformer at position } pos = x + PE_{pos}.
$$

After this addition, "Alice" at position 0 and "Alice" at position 2 are represented by different vectors.

## The sinusoidal positional encoding

**Assumptions.** Each token is represented by a vector of length $d_{\text{model}}$ (the model or embedding dimension), and $d_{\text{model}}$ is even. Tokens are numbered by their position $pos = 0, 1, 2, \dots$ starting at zero.

**Definition** (Section 3.5 of [@VASWANI2017AttentionAllYou]). The positional encoding of position $pos$ is the vector $PE_{pos} \in \mathbb{R}^{d_{\text{model}}}$ with coordinates

$$
\begin{aligned}
PE_{(pos,\, 2i)}   &= \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \\
PE_{(pos,\, 2i+1)} &= \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right),
\end{aligned}
\qquad i = 0, 1, \dots, \frac{d_{\text{model}}}{2} - 1.
$$

The symbols in the formula are:

| Symbol | Meaning |
|--------|---------|
| $pos$ | Position of the token in the sentence: $0$ for the first token, $1$ for the second, ... |
| $d_{\text{model}}$ | Length of the embedding vectors (512 in the paper). The positional encoding has the same length so that the two can be added. |
| $i$ | Index of a *pair* of coordinates. Each $i$ produces two coordinates: one sine and one cosine. |
| $2i$ | Even coordinate (column) of $PE_{pos}$: filled with a sine. |
| $2i + 1$ | Odd coordinate (column) of $PE_{pos}$: filled with a cosine. |
| $10000$ | A constant chosen by the authors. It controls how slowly the last coordinates change. |
| $(pos, j)$ | The $j$-th coordinate of the encoding of position $pos$, i.e. row $pos$, column $j$ of the matrix $PE$. |

It is convenient to give the denominator its own name. For each pair $i$ we define the **(angular) frequency**

$$
\omega_i = \frac{1}{10000^{2i/d_{\text{model}}}} = 10000^{-2i/d_{\text{model}}}.
$$

With this notation the formula becomes

$$
PE_{(pos,\, 2i)} = \sin(pos \cdot \omega_i), \qquad PE_{(pos,\, 2i+1)} = \cos(pos \cdot \omega_i).
$$

So the pair $i$ is a point on the unit circle at angle $pos \cdot \omega_i$. Moving one position forward rotates this point by the angle $\omega_i$.

- For $i = 0$ we get $\omega_0 = 1$: the angle grows fast, the coordinates change a lot from one position to the next.
- For the last pair, $\omega_i$ is close to $1/10000$: the angle grows very slowly.

The **wavelength** (number of positions after which a pair repeats itself) is $2\pi / \omega_i$, which ranges from $2\pi \approx 6.3$ positions (for $i = 0$) up to almost $10000 \cdot 2\pi$ positions. Fast-changing pairs distinguish neighbouring positions, slow-changing pairs distinguish positions that are far apart (similar to the hands of a clock).

## A small example

We use the sentence **"Alice was getting very tired"** and treat each of the five words as one token. To be able to print every number we choose a tiny model dimension $d_{\text{model}} = 4$ (instead of 512). Then $i$ takes the values $0$ and $1$, giving two sine/cosine pairs.

| Word    | Position $pos$ | Index (Numpy) |
|---------|----------------|---------------|
| Alice   | 0              | 0             |
| was     | 1              | 1             |
| getting | 2              | 2             |
| very    | 3              | 3             |
| tired   | 4              | 4             |

The paper counts positions from zero, so here the position and the numpy row index coincide.

In [2]:
sentence = ["Alice", "was", "getting", "very", "tired"]

n_tokens = len(sentence)
d_model = 4

positions = np.arange(n_tokens)
print("Number of tokens:", n_tokens)
print("Positions:", positions)

Number of tokens: 5
Positions: [0 1 2 3 4]


### Step 1: the frequencies $\omega_i$

With $d_{\text{model}} = 4$ there are $d_{\text{model}} / 2 = 2$ frequencies:

$$
\omega_0 = 10000^{-0/4} = 10000^{0} = 1, \qquad
\omega_1 = 10000^{-2/4} = 10000^{-1/2} = \frac{1}{100} = 0.01.
$$

We collect them in the vector

$$
\omega = \begin{bmatrix} \omega_0 \\ \omega_1 \end{bmatrix} = \begin{bmatrix} 1 \\ 0.01 \end{bmatrix}.
$$

In [3]:
pair_index = np.arange(d_model // 2)          # i = 0, 1
frequencies = 1 / 10000 ** (2 * pair_index / d_model)

print("i:      ", pair_index)
print("2i:     ", 2 * pair_index)
print("omega_i:", frequencies)

i:       [0 1]
2i:      [0 2]
omega_i: [1.   0.01]


### Step 2: the angle matrix $A$

Each position is combined with each frequency. We store the angles $pos \cdot \omega_i$ in a matrix $A$ with one row per position and one column per pair $i$:

$$
A = \begin{bmatrix}
0 \cdot \omega_0 & 0 \cdot \omega_1 \\
1 \cdot \omega_0 & 1 \cdot \omega_1 \\
2 \cdot \omega_0 & 2 \cdot \omega_1 \\
3 \cdot \omega_0 & 3 \cdot \omega_1 \\
4 \cdot \omega_0 & 4 \cdot \omega_1
\end{bmatrix}
= \begin{bmatrix}
0 & 0 \\
1 & 0.01 \\
2 & 0.02 \\
3 & 0.03 \\
4 & 0.04
\end{bmatrix}
\in \mathbb{R}^{5 \times 2}.
$$

This is the outer product of the position vector and the frequency vector:

$$
A = \begin{bmatrix} 0 \\ 1 \\ 2 \\ 3 \\ 4 \end{bmatrix}
\begin{bmatrix} \omega_0 & \omega_1 \end{bmatrix}.
$$

In numpy, adding axes with `None` turns the two 1-D arrays into a $5 \times 1$ column and a $1 \times 2$ row, and broadcasting multiplies every position with every frequency.

In [ ]:
angles = positions[:, None] * frequencies[None, :]

pd.DataFrame(
    angles,
    index=[f"{word} (pos={pos})" for pos, word in enumerate(sentence)],
    columns=[f"i={i}" for i in pair_index],
)


,i=0,i=1
Alice (pos=0),0.0,0.00
was (pos=1),1.0,0.01
getting (pos=2),2.0,0.02
very (pos=3),3.0,0.03
tired (pos=4),4.0,0.04


### Step 3: the positional encoding matrix $PE$

We now apply sine and cosine to the angles. The sine values go into the even columns ($2i = 0, 2$) and the cosine values into the odd columns ($2i + 1 = 1, 3$). Stacking the vectors $PE_{pos}$ as rows gives a $5 \times 4$ matrix (one row per token, one column per embedding coordinate):

$$
PE = \begin{bmatrix}
\sin(0 \cdot \omega_0) & \cos(0 \cdot \omega_0) & \sin(0 \cdot \omega_1) & \cos(0 \cdot \omega_1) \\
\sin(1 \cdot \omega_0) & \cos(1 \cdot \omega_0) & \sin(1 \cdot \omega_1) & \cos(1 \cdot \omega_1) \\
\sin(2 \cdot \omega_0) & \cos(2 \cdot \omega_0) & \sin(2 \cdot \omega_1) & \cos(2 \cdot \omega_1) \\
\sin(3 \cdot \omega_0) & \cos(3 \cdot \omega_0) & \sin(3 \cdot \omega_1) & \cos(3 \cdot \omega_1) \\
\sin(4 \cdot \omega_0) & \cos(4 \cdot \omega_0) & \sin(4 \cdot \omega_1) & \cos(4 \cdot \omega_1)
\end{bmatrix}
= \begin{bmatrix}
\sin(0) & \cos(0) & \sin(0) & \cos(0) \\
\sin(1) & \cos(1) & \sin(0.01) & \cos(0.01) \\
\sin(2) & \cos(2) & \sin(0.02) & \cos(0.02) \\
\sin(3) & \cos(3) & \sin(0.03) & \cos(0.03) \\
\sin(4) & \cos(4) & \sin(0.04) & \cos(0.04)
\end{bmatrix}.
$$

The column headers in terms of the formula are $PE_{(pos, 0)}, PE_{(pos, 1)}, PE_{(pos, 2)}, PE_{(pos, 3)}$, i.e. $(2i, 2i+1) = (0, 1)$ for $i = 0$ and $(2i, 2i+1) = (2, 3)$ for $i = 1$.

**Worked row.** For "getting" ($pos = 2$), with angles measured in radians:

$$
PE_{2} = \begin{bmatrix} \sin(2) & \cos(2) & \sin(0.02) & \cos(0.02) \end{bmatrix}
\approx \begin{bmatrix} 0.9093 & -0.4161 & 0.0200 & 0.9998 \end{bmatrix}.
$$

In [5]:
pe_matrix = np.zeros((n_tokens, d_model))
pe_matrix[:, 0::2] = np.sin(angles)   # even columns 0, 2, ...
pe_matrix[:, 1::2] = np.cos(angles)   # odd columns 1, 3, ...

pe_table = pd.DataFrame(
    pe_matrix,
    index=sentence,
    columns=[f"dim {j}" for j in range(d_model)],
)
pe_table.round(4)

,dim 0,dim 1,dim 2,dim 3
Alice,0.0000,1.0000,0.00,1.0000
was,0.8415,0.5403,0.01,1.0000
getting,0.9093,-0.4161,0.02,0.9998
very,0.1411,-0.9900,0.03,0.9996
tired,-0.7568,-0.6536,0.04,0.9992


The row for "getting" matches the values computed by hand. Note also the first row: position $0$ always gets the vector $(0, 1, 0, 1, \dots)$ because $\sin(0) = 0$ and $\cos(0) = 1$.

To make sure that the vectorised code really implements the formula, we write a second version with explicit loops that follows the definition line by line and compare the two.

In [6]:
def positional_encoding(n_positions, d_model):
    # Vectorised sinusoidal positional encoding (rows = positions, columns = dimensions).
    positions = np.arange(n_positions)
    frequencies = 1 / 10000 ** (2 * np.arange(d_model // 2) / d_model)
    angles = positions[:, None] * frequencies[None, :]
    encoding = np.zeros((n_positions, d_model))
    encoding[:, 0::2] = np.sin(angles)
    encoding[:, 1::2] = np.cos(angles)
    return encoding


def positional_encoding_loops(n_positions, d_model):
    # The formula from the paper, one entry at a time.
    encoding = np.zeros((n_positions, d_model))
    for pos in range(n_positions):
        for i in range(d_model // 2):
            angle = pos / 10000 ** (2 * i / d_model)
            encoding[pos, 2 * i] = np.sin(angle)
            encoding[pos, 2 * i + 1] = np.cos(angle)
    return encoding


assert np.allclose(positional_encoding(n_tokens, d_model), pe_matrix)
assert np.allclose(positional_encoding_loops(n_tokens, d_model), pe_matrix)
print("Both implementations agree with the PE matrix above.")

Both implementations agree with the PE matrix above.


### Step 4: adding the encodings to the word embeddings

Let $X \in \mathbb{R}^{5 \times 4}$ be the matrix of word embeddings: row $pos$ is the embedding $x_{pos}$ of the word at position $pos$. In a real Transformer these embeddings are learned during training (just like the Word2Vec weights). Here we fill $X$ with random numbers.

The input to the first Transformer layer is the sum

$$
Z = X + PE, \qquad z_{pos} = x_{pos} + PE_{pos},
$$

$$
\underbrace{\begin{bmatrix}
z_{0,0} & z_{0,1} & z_{0,2} & z_{0,3} \\
\vdots & \vdots & \vdots & \vdots \\
z_{4,0} & z_{4,1} & z_{4,2} & z_{4,3}
\end{bmatrix}}_{Z \; (5 \times 4)}
=
\underbrace{\begin{bmatrix}
x_{0,0} & x_{0,1} & x_{0,2} & x_{0,3} \\
\vdots & \vdots & \vdots & \vdots \\
x_{4,0} & x_{4,1} & x_{4,2} & x_{4,3}
\end{bmatrix}}_{X \; (5 \times 4)}
+
\underbrace{\begin{bmatrix}
PE_{(0,0)} & PE_{(0,1)} & PE_{(0,2)} & PE_{(0,3)} \\
\vdots & \vdots & \vdots & \vdots \\
PE_{(4,0)} & PE_{(4,1)} & PE_{(4,2)} & PE_{(4,3)}
\end{bmatrix}}_{PE \; (5 \times 4)}.
$$

This is why the positional encoding must have the same length $d_{\text{model}}$ as the embeddings. (In the paper the embeddings are additionally multiplied by $\sqrt{d_{\text{model}}}$ before the addition; we skip this scaling here.)

In [ ]:
np.random.seed(0)

word_embeddings = np.random.randn(n_tokens, d_model)

embedding_table = pd.DataFrame(
    word_embeddings, index=sentence, columns=[f"dim {j}" for j in range(d_model)]
)
print("Word embeddings X:")
embedding_table.round(4)

In [ ]:
transformer_input = word_embeddings + pe_matrix

input_table = pd.DataFrame(
    transformer_input, index=sentence, columns=[f"dim {j}" for j in range(d_model)]
)
print("Z = X + PE:")
input_table.round(4)

Now suppose the word "Alice" appeared both at position 0 and at position 4. Its word embedding would be identical in both places, but the inputs $z_0$ and $z_4$ differ by $PE_4 - PE_0$:

In [ ]:
alice_embedding = word_embeddings[0]

alice_at_pos_0 = alice_embedding + pe_matrix[0]
alice_at_pos_4 = alice_embedding + pe_matrix[4]

print("Alice at position 0:", alice_at_pos_0.round(4))
print("Alice at position 4:", alice_at_pos_4.round(4))
print("Difference PE_4 - PE_0:", (pe_matrix[4] - pe_matrix[0]).round(4))

## Visualising the encodings

The heatmap shows the $5 \times 4$ matrix $PE$ from the example. The first two columns ($i = 0$, $\omega_0 = 1$) change a lot from row to row, while the last two columns ($i = 1$, $\omega_1 = 0.01$) barely move over five positions.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(pe_matrix, cmap="RdBu", vmin=-1, vmax=1)

for row in range(n_tokens):
    for col in range(d_model):
        ax.text(col, row, f"{pe_matrix[row, col]:.2f}", ha="center", va="center")

ax.set_xticks(range(d_model), [f"dim {j}" for j in range(d_model)])
ax.set_yticks(range(n_tokens), [f"{word} ({pos})" for pos, word in enumerate(sentence)])
ax.set_title("Positional encodings, $d_{model} = 4$")
fig.colorbar(image, ax=ax)
plt.show()

To see the waves behind these numbers, we evaluate the same four columns for positions $0, 1, \dots, 50$. The dots mark our five tokens. With $\omega_0 = 1$ the first pair completes a full cycle every $2\pi \approx 6.3$ positions; with $\omega_1 = 0.01$ the second pair would need about $628$ positions.

In [ ]:
many_positions = np.arange(51)
pe_long = positional_encoding(len(many_positions), d_model)

fig, axes = plt.subplots(d_model, 1, figsize=(8, 7), sharex=True)
for dim, ax in enumerate(axes):
    pair = dim // 2
    function_name = "sin" if dim % 2 == 0 else "cos"
    ax.plot(many_positions, pe_long[:, dim])
    ax.scatter(positions, pe_matrix[:, dim], color="black", zorder=3)
    ax.set_ylim(-1.1, 1.1)
    ax.set_ylabel(f"dim {dim}")
    ax.set_title(f"dim {dim} = {function_name}(pos * omega_{pair})", fontsize=10)
axes[-1].set_xlabel("position pos")
plt.tight_layout()
plt.show()

With a realistic dimension the familiar pattern appears. Below, $d_{\text{model}} = 64$ and 50 positions: the left columns oscillate quickly, the right columns change slowly, so every row (position) gets a distinct pattern.

In [ ]:
pe_large = positional_encoding(50, 64)

fig, ax = plt.subplots(figsize=(9, 5))
image = ax.imshow(pe_large, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
ax.set_xlabel("embedding dimension j")
ax.set_ylabel("position pos")
ax.set_title("Positional encodings, $d_{model} = 64$")
fig.colorbar(image, ax=ax)
plt.show()

## Properties of the sinusoidal encoding

### Property 1: bounded values

**Statement.** Every entry satisfies $-1 \le PE_{(pos, j)} \le 1$.

**Proof.** Every entry is a sine or a cosine of a real number. $\square$

This keeps the encodings on a scale comparable to the word embeddings, no matter how long the sentence is (in contrast to, e.g., adding the raw position number $pos$).

In [ ]:
print("Minimum entry of pe_large:", pe_large.min().round(4))
print("Maximum entry of pe_large:", pe_large.max().round(4))

### Property 2: shifting the position is a rotation

The authors chose sine and cosine because "for any fixed offset $k$, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$" [@VASWANI2017AttentionAllYou]. The hope is that this makes it easy for the model to attend by *relative* position ("the word two places to the left").

**Statement.** Fix a pair $i$ and an offset $k$. Then

$$
\begin{bmatrix} PE_{(pos+k,\, 2i)} \\ PE_{(pos+k,\, 2i+1)} \end{bmatrix}
=
\underbrace{\begin{bmatrix} \cos(k\omega_i) & \sin(k\omega_i) \\ -\sin(k\omega_i) & \cos(k\omega_i) \end{bmatrix}}_{R_i(k)}
\begin{bmatrix} PE_{(pos,\, 2i)} \\ PE_{(pos,\, 2i+1)} \end{bmatrix}.
$$

The matrix $R_i(k)$ is a rotation matrix. It depends on the offset $k$ and the frequency $\omega_i$, but **not** on the position $pos$.

**Proof.** Write $a = pos \cdot \omega_i$ and $b = k \cdot \omega_i$, so that $(pos + k)\omega_i = a + b$. The angle addition formulas give

$$
\begin{aligned}
\sin(a + b) &= \cos(b) \sin(a) + \sin(b) \cos(a), \\
\cos(a + b) &= -\sin(b) \sin(a) + \cos(b) \cos(a),
\end{aligned}
$$

which is exactly the matrix product above with $\sin(a) = PE_{(pos, 2i)}$ and $\cos(a) = PE_{(pos, 2i+1)}$. $\square$

Applying $R_0(k), R_1(k), \dots$ to all pairs at once gives a block-diagonal $d_{\text{model}} \times d_{\text{model}}$ matrix $M(k)$ with $PE_{pos+k} = M(k)\, PE_{pos}$. For our example with $d_{\text{model}} = 4$:

$$
M(k) = \begin{bmatrix}
\cos(k\omega_0) & \sin(k\omega_0) & 0 & 0 \\
-\sin(k\omega_0) & \cos(k\omega_0) & 0 & 0 \\
0 & 0 & \cos(k\omega_1) & \sin(k\omega_1) \\
0 & 0 & -\sin(k\omega_1) & \cos(k\omega_1)
\end{bmatrix}.
$$

We check this numerically with $k = 2$: the same matrix $M(2)$ maps "Alice" ($pos = 0$) to "getting" ($pos = 2$), "was" ($pos = 1$) to "very" ($pos = 3$), and "getting" ($pos = 2$) to "tired" ($pos = 4$).

In [ ]:
def shift_matrix(offset, frequencies):
    # Block-diagonal matrix M(k) built from one 2x2 rotation per frequency.
    d_model = 2 * len(frequencies)
    matrix = np.zeros((d_model, d_model))
    for i, omega in enumerate(frequencies):
        angle = offset * omega
        matrix[2 * i:2 * i + 2, 2 * i:2 * i + 2] = [
            [np.cos(angle), np.sin(angle)],
            [-np.sin(angle), np.cos(angle)],
        ]
    return matrix


offset = 2
shift_by_2 = shift_matrix(offset, frequencies)
print("M(2):")
print(shift_by_2.round(4))

In [ ]:
for pos in range(n_tokens - offset):
    shifted = shift_by_2 @ pe_matrix[pos]
    target = pe_matrix[pos + offset]
    print(f"M(2) @ PE_{pos} ({sentence[pos]:>7}) = {shifted.round(4)}  "
          f"PE_{pos + offset} ({sentence[pos + offset]:>7}) = {target.round(4)}  "
          f"equal: {np.allclose(shifted, target)}")

### Property 3: similarity depends only on the distance

**Statement.** For any positions $pos$ and $pos + k$,

$$
PE_{pos} \cdot PE_{pos+k} = \sum_{i=0}^{d_{\text{model}}/2 - 1} \cos(k\, \omega_i).
$$

**Proof.** The dot product is a sum over pairs. For one pair, with $a = pos \cdot \omega_i$ and $b = (pos + k)\omega_i$,

$$
\sin(a)\sin(b) + \cos(a)\cos(b) = \cos(b - a) = \cos(k\,\omega_i). \qquad \square
$$

Consequently the dot product of two encodings depends only on their distance $k$, not on where in the sentence they are. In particular $PE_{pos} \cdot PE_{pos} = d_{\text{model}}/2$, so all encodings have the same length $\sqrt{d_{\text{model}}/2}$.

For our sentence we collect all dot products in the $5 \times 5$ matrix $G = PE \, PE^{\top}$ with entries $G_{pos, pos'} = PE_{pos} \cdot PE_{pos'}$. By the statement above, $G$ is constant along each diagonal (all entries with the same distance $|pos - pos'|$ are equal), and its main diagonal is $d_{\text{model}} / 2 = 2$.

In [ ]:
gram_matrix = pe_matrix @ pe_matrix.T

pd.DataFrame(gram_matrix, index=sentence, columns=sentence).round(4)

In [ ]:
for distance in range(n_tokens):
    formula_value = np.cos(distance * frequencies).sum()
    diagonal_values = np.diag(gram_matrix, k=distance)
    print(f"k = {distance}: formula = {formula_value:.4f}, "
          f"entries on diagonal = {diagonal_values.round(4)}")

## Summary

- Self-attention by itself ignores word order, so the Transformer adds a position-dependent vector $PE_{pos}$ to each word embedding: $z_{pos} = x_{pos} + PE_{pos}$.
- The sinusoidal encoding uses $d_{\text{model}}/2$ frequencies $\omega_i = 10000^{-2i/d_{\text{model}}}$. Coordinate $2i$ holds $\sin(pos\,\omega_i)$, coordinate $2i+1$ holds $\cos(pos\,\omega_i)$.
- Computing $PE$ takes three steps: the frequency vector $\omega$, the angle matrix $A = pos \, \omega^{\top}$, and $\sin$/$\cos$ of $A$ placed in the even/odd columns.
- The encodings are bounded, a shift by $k$ positions is a fixed rotation $M(k)$, and the dot product of two encodings depends only on their distance.
- The encodings contain no learned parameters. The paper reports that learned positional embeddings performed nearly identically, and chose the sinusoidal version because it may extrapolate to sentences longer than those seen in training [@VASWANI2017AttentionAllYou].

## Why sine *and* cosine pairs?

The formula uses every frequency $\omega_i$ twice: once inside a sine (coordinate $2i$) and once inside a cosine (coordinate $2i+1$). This section explains why a single sine per frequency would not be enough, and whether it matters that the sine goes to the even and the cosine to the odd coordinate.

Throughout, fix one pair $i$ and write $\theta = pos \cdot \omega_i$ for its angle. The pair is the point

$$
\begin{bmatrix} PE_{(pos,\, 2i)} \\ PE_{(pos,\, 2i+1)} \end{bmatrix}
= \begin{bmatrix} \sin(\theta) \\ \cos(\theta) \end{bmatrix}
$$

on the unit circle, since $\sin^2(\theta) + \cos^2(\theta) = 1$.

### Reason 1: the pair identifies the angle, a single sine does not

**Statement.** The pair $(\sin\theta, \cos\theta)$ determines $\theta$ uniquely up to multiples of $2\pi$. The sine alone does not: $\sin(\theta) = \sin(\pi - \theta)$, so two different angles share every sine value in $(-1, 1)$.

**Proof sketch.** A point on the unit circle corresponds to exactly one angle in $(-\pi, \pi]$. Knowing only $\sin\theta$ fixes the height of the point, which is shared by two points (left and right half of the circle). The cosine tells us which half. $\square$

In numpy, `np.arcsin` inverts the sine, but can only return angles in $[-\pi/2, \pi/2]$. `np.arctan2(sin, cos)` uses both coordinates and returns the angle in $(-\pi, \pi]$. For the first pair of our example ($\omega_0 = 1$, so $\theta = pos$):

In [ ]:
angle_first_pair = positions * frequencies[0]
sine_values = pe_matrix[:, 0]
cosine_values = pe_matrix[:, 1]

pd.DataFrame(
    {
        "true angle": angle_first_pair,
        "true angle mod 2pi (in (-pi, pi])": np.angle(np.exp(1j * angle_first_pair)),
        "arcsin(sin) (sine only)": np.arcsin(sine_values),
        "arctan2(sin, cos) (pair)": np.arctan2(sine_values, cosine_values),
    },
    index=sentence,
).round(4)

From the sine alone we would conclude that "getting" ($\theta = 2$) has angle $\pi - 2 \approx 1.14$, i.e. "getting" would look as if it came just after "was" (angle $1.14$ instead of $2$), and "very" ($\theta = 3$) would even get a smaller angle than "was" ($0.14$). The pair recovers the correct angle for every position (for "tired", $4 - 2\pi \approx -2.28$ is the same point on the circle as $4$).

The same ambiguity can be seen directly: positions 1 and 2 have almost the same sine, but their cosines have opposite signs. Only the cosine tells them apart.

In [ ]:
long_positions = np.arange(50)
sine_only = np.sin(long_positions * frequencies[0])
cosine_only = np.cos(long_positions * frequencies[0])

# Sine and cosine of the first pair (omega_0 = 1) for neighbouring positions 1 and 2
for pos in [1, 2]:
    print(f"pos {pos}: sin = {sine_only[pos]:.3f}, cos = {cosine_only[pos]:.3f}")

### Reason 2: a shift by $k$ positions is linear only for the pair

In Property 2 we showed $PE_{pos+k} = M(k)\, PE_{pos}$ with a matrix $M(k)$ that does not depend on $pos$. The proof needed both the sine and the cosine of $\theta$:

$$
\sin(\theta + k\omega_i) = \cos(k\omega_i)\, \sin(\theta) + \sin(k\omega_i)\, \cos(\theta).
$$

If we only stored $\sin(\theta)$, we would need a single number $c$ (a $1 \times 1$ "matrix") with $\sin(\theta + k\omega_i) = c \sin(\theta)$ for all positions. No such $c$ exists: at $pos = 0$ the right-hand side is $c \cdot 0 = 0$, while the left-hand side is $\sin(k\omega_i)$, which is not zero in general.

We check this numerically by fitting the best linear map (least squares) from the encoding at position $pos$ to the encoding at position $pos + 2$, using positions $0, \dots, 47$ and the first frequency $\omega_0 = 1$:

- *sine only*: predict $\sin((pos + 2)\omega_0)$ from $\sin(pos\,\omega_0)$ ($1 \times 1$ matrix),
- *pair*: predict $(\sin, \cos)$ at $pos + 2$ from $(\sin, \cos)$ at $pos$ ($2 \times 2$ matrix).

In [ ]:
offset = 2
current = np.arange(48)
target = current + offset
omega = frequencies[0]

# Sine only: one coordinate per frequency
sine_now = np.sin(current * omega)[:, None]
sine_later = np.sin(target * omega)[:, None]
sine_map, *_ = np.linalg.lstsq(sine_now, sine_later, rcond=None)
sine_error = np.abs(sine_now @ sine_map - sine_later).max()

# Pair: sine and cosine of the same angle
pair_now = np.column_stack([np.sin(current * omega), np.cos(current * omega)])
pair_later = np.column_stack([np.sin(target * omega), np.cos(target * omega)])
pair_map, *_ = np.linalg.lstsq(pair_now, pair_later, rcond=None)
pair_error = np.abs(pair_now @ pair_map - pair_later).max()

print("Sine only: best 1x1 map =", sine_map.round(4).ravel(), " largest error =", sine_error.round(4))
print("Pair:      best 2x2 map =")
print(pair_map.T.round(4))
print("           largest error =", pair_error.round(10))

With the pair the fitted map is exactly the rotation block of $M(2)$ (compare with the upper-left $2 \times 2$ block printed in Property 2) and the error is zero. With the sine alone the best linear map still makes errors of the size of the values themselves.

### Reason 3: every pair contributes the same length

Because $\sin^2(\theta) + \cos^2(\theta) = 1$, each pair adds exactly $1$ to the squared length of $PE_{pos}$:

$$
\lVert PE_{pos} \rVert^2 = \sum_{i=0}^{d_{\text{model}}/2 - 1} \left( \sin^2(pos\,\omega_i) + \cos^2(pos\,\omega_i) \right) = \frac{d_{\text{model}}}{2}.
$$

So no position gets a "stronger" encoding than another. With sines only, the length would depend on the position (position $0$ would even get the zero vector).

In [ ]:
pd.DataFrame(
    {
        "length with pairs": np.linalg.norm(pe_matrix, axis=1),
        "length with sines only": np.linalg.norm(pe_matrix[:, 0::2], axis=1),
    },
    index=sentence,
).round(4)

### Why sine on even and cosine on odd coordinates?

This choice is a **convention**, not a mathematical requirement. What matters is that

1. each frequency $\omega_i$ appears in both a sine and a cosine, and
2. every position uses the *same* assignment of these values to coordinates.

Interleaving (sine at $2i$, cosine at $2i+1$) keeps the two members of a pair next to each other, which makes the formula compact and gives the block-diagonal shift matrix $M(k)$ above. Other implementations instead put all sines in the first half and all cosines in the second half,

$$
\left[\sin(pos\,\omega_0), \dots, \sin(pos\,\omega_{d/2-1}),\ \cos(pos\,\omega_0), \dots, \cos(pos\,\omega_{d/2-1})\right],
$$

and work just as well.

**Statement.** The two layouts differ only by a fixed permutation of the coordinates: $PE^{\text{concat}} = PE^{\text{interleaved}}\, P$ for a permutation matrix $P$ that is the same for all positions. Since the first thing the Transformer does with its input is multiply it by learned weight matrices $W$, and

$$
PE^{\text{concat}}\, W = PE^{\text{interleaved}}\, (P W),
$$

any model trained with one layout can be turned into an equivalent model for the other layout by reordering the rows of $W$. (Adding the word embeddings does not change this: they are learned too, so their coordinates can be reordered in the same way.) $\square$

We build both layouts for our example and verify this.

In [ ]:
pe_concat = np.hstack([np.sin(angles), np.cos(angles)])

# Column order of the concatenated layout, expressed as columns of the interleaved layout
concat_order = list(range(0, d_model, 2)) + list(range(1, d_model, 2))
permutation = np.eye(d_model)[:, concat_order]

print("Concatenated layout takes interleaved columns", concat_order)
print("Permutation matrix P:")
print(permutation.astype(int))
print("PE_concat == PE_interleaved @ P:", np.allclose(pe_concat, pe_matrix @ permutation))

In [ ]:
np.random.seed(1)
weights = np.random.randn(d_model, 3)   # a learned linear layer with 3 outputs

output_concat = pe_concat @ weights
output_interleaved = pe_matrix @ (permutation @ weights)

print("Same layer output for both layouts:", np.allclose(output_concat, output_interleaved))

**In short:** the cosine is paired with the sine so that each frequency is described by a full point on the circle. This makes positions identifiable, makes shifts by $k$ positions a linear (rotation) map, and gives every position the same length. Which coordinate receives the sine and which the cosine is only a bookkeeping convention.